# Experiment 16: Domain Segmentation — Refined

**Building on Experiment 15** which showed domain segmentation + per-domain threshold optimisation gives +3 pp F1 (51.88% → 54.89%).

**Problems with Exp 15 to fix:**
- Thresholds were tuned **on the test set** — this leaks. True gain is unknown.
- All domains share the same hyperparameters — small domains likely over/under-regularised.
- All domains use the same feature set — some features are domain-irrelevant noise.
- Domain signal was never fed back into the universal model.

**Refinements in this notebook:**
1. **Proper train/val/test split** — 2017 carved out as validation; thresholds/calibration tuned there.
2. **Calibrated probabilities** — Platt scaling per domain (more principled than threshold sweeping).
3. **Per-domain feature selection** — SelectKBest(k=50) per domain before fitting LightGBM.
4. **Per-domain hyperparameter tuning** — lightweight grid search on val set per domain.
5. **Domain-as-feature** — add domain OHE to the universal model as a soft signal.
6. **Final head-to-head comparison** of all variants on the held-out 2018-2020 test set.

In [ ]:
import sys
sys.path.append('../../')

import copy
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    accuracy_score, roc_auc_score, confusion_matrix
)
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# ── helper ──────────────────────────────────────────────────────────────
def metrics(y_true, y_pred, y_proba=None):
    out = dict(
        Accuracy  = accuracy_score(y_true, y_pred),
        Precision = precision_score(y_true, y_pred, zero_division=0),
        Recall    = recall_score(y_true, y_pred, zero_division=0),
        F1        = f1_score(y_true, y_pred, zero_division=0),
    )
    if y_proba is not None and len(np.unique(y_true)) > 1:
        out['ROC-AUC'] = roc_auc_score(y_true, y_proba)
    return out

def best_threshold(y_true, y_proba, lo=0.30, hi=0.75, step=0.01):
    """Find threshold maximising F1 on the supplied labels."""
    best_f1, best_t = 0.0, 0.54
    for t in np.arange(lo, hi, step):
        f1 = f1_score(y_true, (y_proba >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1

print('Imports OK')

## 1. Load Data

In [ ]:
df      = pd.read_pickle('../../data/processed/cleaned_data.pkl')
X_all   = pd.read_pickle('../../data/features/X_all.pkl')
y_cls   = pd.read_pickle('../../data/features/y_classification.pkl')

print(f"Dataset : {df.shape}")
print(f"Features: {X_all.shape}")
print(f"Target  : {y_cls.shape}  (positive rate: {y_cls.mean()*100:.1f}%)")
print(f"Years   : {df['Year'].min()} – {df['Year'].max()}")
print()
print(df['Year'].value_counts().sort_index())

## 2. Domain Mapping

In [ ]:
# ── locate ASJC column ──────────────────────────────────────────────────
asjc_col = next(
    (c for c in df.columns if 'asjc' in c.lower()),
    None
)
print(f"ASJC column: '{asjc_col}'")

def map_to_domain(val):
    if pd.isna(val):
        return 'Other'
    s = str(val).lower()
    if 'multidisciplinary' in s:
        return 'Multidisciplinary'
    medicine = [
        'medicine','surgery','nursing','health','cardiology','cardiovascular',
        'oncology','cancer','radiology','nuclear medicine','anesthesiology',
        'obstetrics','gynecology','urology','ophthalmology','hematology',
        'epidemiology','emergency','gastroenterology','hepatology',
        'rheumatology','orthopedic','dermatology','psychiatry','neurology',
        'pediatrics','otorhinolaryngology','infectious diseases','pulmonary',
        'respiratory','critical care','intensive care','pharmacology',
        'immunology','allergy','transplantation','pathology','anatomy',
        'physiology','physical therapy','rehabilitation','dentistry',
        'endocrinology','nephrology','geriatrics','palliative',
        'clinical','medical','hospital','patient','diagnosis','treatment',
        'microbiology (medical)','genetics','general nursing',
    ]
    engineering = [
        'engineering','electrical','electronic','mechanical','civil',
        'chemical engineering','aerospace','biomedical engineering',
        'industrial','manufacturing','control and systems','automation',
        'telecommunications','signal processing','computer science',
        'information systems','software','hardware','artificial intelligence',
        'machine learning','computational','materials science','energy',
        'renewable energy','nuclear energy','robotics','mechatronics',
    ]
    social = [
        'education','psychology','economics','business','management',
        'social science','communication','policy','political','sociology',
        'anthropology','history','philosophy','linguistics','law',
        'public administration','cultural','media','journalism',
        'library','information science','tourism','sport','geography',
        'demography','urban','development studies','gender','religion',
        'arts and humanities','architecture','urban planning',
        'accounting','finance','marketing','strategy','organizational',
        'human resource','supply chain','operations research',
        'health (social science)',
    ]
    natural = [
        'chemistry','physics','mathematics','biology','biochemistry',
        'molecular biology','cellular','ecology','evolution','botany',
        'zoology','marine','oceanography','atmospheric','geology',
        'geoscience','astronomy','astrophysics','biophysics',
        'organic chemistry','inorganic chemistry',
        'physical and theoretical chemistry','spectroscopy','catalysis',
        'colloid','surface chemistry','analytical chemistry',
        'nature and landscape','environmental science','earth',
        'planetary','agricultural','food science','nutrition',
        'forestry','aquatic','microbiology (non-medical)',
    ]
    for terms, label in [(medicine,'Medicine & Health'),
                          (engineering,'Engineering & Technology'),
                          (social,'Social Sciences'),
                          (natural,'Natural Sciences')]:
        if any(t in s for t in terms):
            return label
    return 'Other'

if asjc_col:
    df['domain'] = df[asjc_col].apply(map_to_domain)
else:
    df['domain'] = 'Other'

print(df['domain'].value_counts())
DOMAINS = sorted(df['domain'].unique().tolist())

## 3. Train / Val / Test Split

| Set   | Years     | Purpose                                    |
|-------|-----------|--------------------------------------------|
| Train | 2010–2016 | Fit all models                             |
| **Val**   | **2017**      | **Tune thresholds, calibration, hyperparams**  |
| Test  | 2018–2020 | Final held-out evaluation — **never tuned on** |

Carving out 2017 as validation prevents the threshold-leakage problem in Exp 15.

In [ ]:
def make_split(years):
    mask = df['Year'].isin(years)
    idx  = df[mask].index.intersection(X_all.index)
    return X_all.loc[idx], y_cls.loc[idx], df.loc[idx, 'domain']

X_train, y_train, d_train = make_split(range(2010, 2017))
X_val,   y_val,   d_val   = make_split([2017])
X_test,  y_test,  d_test  = make_split([2018, 2019, 2020])

print(f"Train : {X_train.shape}  pos={y_train.mean()*100:.1f}%")
print(f"Val   : {X_val.shape}    pos={y_val.mean()*100:.1f}%")
print(f"Test  : {X_test.shape}   pos={y_test.mean()*100:.1f}%")
print()
print("Train domain dist:")
print(d_train.value_counts())
print()
print("Val domain dist:")
print(d_val.value_counts())

In [ ]:
# Impute NaNs from pre-2015 papers
train_medians = X_train.median()

X_train = X_train.fillna(train_medians).fillna(0)
X_val   = X_val.fillna(train_medians).fillna(0)
X_test  = X_test.fillna(train_medians).fillna(0)

remaining = X_train.isna().sum().sum() + X_val.isna().sum().sum() + X_test.isna().sum().sum()
print(f"Remaining NaNs after imputation: {remaining}")

## 4. Baseline — Universal LightGBM (threshold tuned on val)

This is the fair baseline: fit on train, tune threshold on **val**, report on **test**.

In [ ]:
BASE_CLF = LGBMClassifier(
    n_estimators=500, learning_rate=0.05, num_leaves=63,
    class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1
)

base_clf = copy.deepcopy(BASE_CLF)
base_clf.fit(X_train, y_train)

proba_val_base  = base_clf.predict_proba(X_val)[:, 1]
proba_test_base = base_clf.predict_proba(X_test)[:, 1]

# Tune threshold on val
t_base, f1_val_base = best_threshold(y_val, proba_val_base)
print(f"Universal LightGBM — val threshold: {t_base:.2f}  val F1: {f1_val_base*100:.2f}%")

y_pred_base = (proba_test_base >= t_base).astype(int)
res_base = metrics(y_test, y_pred_base, proba_test_base)

print("\n=== BASELINE TEST RESULTS ===")
for k, v in res_base.items():
    print(f"  {k:<12}: {v*100:.2f}%")

BASELINE_F1 = res_base['F1']

## 5. Refinement 1 — Calibrated Domain Models

Per-domain LightGBM fitted on train, then **Platt-calibrated on val**.
The calibrated probabilities are used at a fixed 0.5 threshold — no test-set tuning.

In [ ]:
print("=" * 80)
print("REFINEMENT 1: CALIBRATED DOMAIN MODELS")
print("=" * 80)

calibrated_models  = {}   # domain -> CalibratedClassifierCV
calib_thresholds   = {}   # domain -> threshold tuned on val
calib_val_results  = {}

SKIP_DOMAINS_CALIB = set()

for domain in DOMAINS:
    tr_mask = d_train == domain
    va_mask = d_val   == domain
    te_mask = d_test  == domain

    X_tr_d, y_tr_d = X_train[tr_mask], y_train[tr_mask]
    X_va_d, y_va_d = X_val[va_mask],   y_val[va_mask]

    if len(X_tr_d) < 50 or len(X_va_d) < 30:
        print(f"  {domain:<30}  SKIP (train={len(X_tr_d)}, val={len(X_va_d)})")
        SKIP_DOMAINS_CALIB.add(domain)
        continue
    if y_tr_d.sum() < 5 or (len(y_tr_d) - y_tr_d.sum()) < 5:
        print(f"  {domain:<30}  SKIP (class imbalance too severe)")
        SKIP_DOMAINS_CALIB.add(domain)
        continue

    # Fit base estimator
    clf = copy.deepcopy(BASE_CLF)
    clf.fit(X_tr_d, y_tr_d)

    # Platt calibration using val set (cv='prefit')
    cal = CalibratedClassifierCV(clf, cv='prefit', method='sigmoid')
    cal.fit(X_va_d, y_va_d)
    calibrated_models[domain] = cal

    # Tune threshold on val calibrated probs
    proba_va_cal = cal.predict_proba(X_va_d)[:, 1]
    t_d, f1_val_d = best_threshold(y_va_d, proba_va_cal)
    calib_thresholds[domain] = t_d
    calib_val_results[domain] = f1_val_d

    print(f"  {domain:<30}  train={len(X_tr_d):4d}  val={len(X_va_d):3d}  "
          f"val_thresh={t_d:.2f}  val_F1={f1_val_d*100:.2f}%")

print(f"\nCalibrated domains: {len(calibrated_models)} / {len(DOMAINS)}")

In [ ]:
# Evaluate calibrated domain models on test set
y_pred_calib       = y_pred_base.copy().astype(float)
y_proba_calib      = proba_test_base.copy()

for domain, cal in calibrated_models.items():
    te_mask = d_test == domain
    if te_mask.sum() < 30:
        continue
    proba_d = cal.predict_proba(X_test[te_mask])[:, 1]
    thresh  = calib_thresholds[domain]
    y_pred_calib[te_mask]  = (proba_d >= thresh).astype(int)
    y_proba_calib[te_mask] = proba_d

res_calib = metrics(y_test, y_pred_calib.astype(int), y_proba_calib)
print("=== CALIBRATED DOMAIN MODELS — TEST ===")
for k, v in res_calib.items():
    diff = v - res_base[k]
    print(f"  {k:<12}: {v*100:.2f}%  ({diff*100:+.2f} pp vs baseline)")

## 6. Refinement 2 — Per-Domain Feature Selection

Select the top-50 features by ANOVA F-score **within each domain's training set**, then re-fit LightGBM on that reduced feature space.

In [ ]:
print("=" * 80)
print("REFINEMENT 2: PER-DOMAIN FEATURE SELECTION  (SelectKBest k=50)")
print("=" * 80)

K_FEATURES = 50

featsel_models     = {}   # domain -> (selector, clf)
featsel_thresholds = {}

for domain in DOMAINS:
    tr_mask = d_train == domain
    va_mask = d_val   == domain

    X_tr_d, y_tr_d = X_train[tr_mask], y_train[tr_mask]
    X_va_d, y_va_d = X_val[va_mask],   y_val[va_mask]

    if len(X_tr_d) < 50 or len(X_va_d) < 30:
        continue
    if y_tr_d.sum() < 5 or (len(y_tr_d) - y_tr_d.sum()) < 5:
        continue

    k = min(K_FEATURES, X_tr_d.shape[1])
    sel = SelectKBest(f_classif, k=k)
    X_tr_sel = sel.fit_transform(X_tr_d, y_tr_d)
    X_va_sel = sel.transform(X_va_d)

    clf = copy.deepcopy(BASE_CLF)
    clf.fit(X_tr_sel, y_tr_d)

    proba_va = clf.predict_proba(X_va_sel)[:, 1]
    t_d, f1_val_d = best_threshold(y_va_d, proba_va)

    featsel_models[domain]     = (sel, clf)
    featsel_thresholds[domain] = t_d

    top_features = X_train.columns[sel.get_support()].tolist()[:5]
    print(f"  {domain:<30}  k={k}  val_thresh={t_d:.2f}  val_F1={f1_val_d*100:.2f}%")
    print(f"    top features: {top_features}")

print(f"\nFeature-selected domain models: {len(featsel_models)}")

In [ ]:
# Evaluate feature-selected models on test set
y_pred_featsel  = y_pred_base.copy().astype(float)
y_proba_featsel = proba_test_base.copy()

for domain, (sel, clf) in featsel_models.items():
    te_mask = d_test == domain
    if te_mask.sum() < 30:
        continue
    X_te_sel = sel.transform(X_test[te_mask])
    proba_d  = clf.predict_proba(X_te_sel)[:, 1]
    thresh   = featsel_thresholds[domain]
    y_pred_featsel[te_mask]  = (proba_d >= thresh).astype(int)
    y_proba_featsel[te_mask] = proba_d

res_featsel = metrics(y_test, y_pred_featsel.astype(int), y_proba_featsel)
print("=== FEATURE-SELECTED DOMAIN MODELS — TEST ===")
for k, v in res_featsel.items():
    diff = v - res_base[k]
    print(f"  {k:<12}: {v*100:.2f}%  ({diff*100:+.2f} pp vs baseline)")

## 7. Refinement 3 — Per-Domain Hyperparameter Tuning

Grid search over a small set of LightGBM configs, scored on **val F1** per domain.
Small domains get stronger regularisation; large domains can afford deeper trees.

In [ ]:
print("=" * 80)
print("REFINEMENT 3: PER-DOMAIN HYPERPARAMETER TUNING")
print("=" * 80)

# Four configs spanning regularisation strength and tree depth
HP_GRID = [
    dict(n_estimators=200, learning_rate=0.05, num_leaves=31,  min_child_samples=20),  # conservative
    dict(n_estimators=300, learning_rate=0.05, num_leaves=63,  min_child_samples=10),  # default-ish
    dict(n_estimators=500, learning_rate=0.03, num_leaves=127, min_child_samples=5),   # expressive
    dict(n_estimators=200, learning_rate=0.10, num_leaves=31,  min_child_samples=30),  # fast+regularised
]

tuned_models     = {}
tuned_thresholds = {}

for domain in DOMAINS:
    tr_mask = d_train == domain
    va_mask = d_val   == domain

    X_tr_d, y_tr_d = X_train[tr_mask], y_train[tr_mask]
    X_va_d, y_va_d = X_val[va_mask],   y_val[va_mask]

    if len(X_tr_d) < 50 or len(X_va_d) < 30:
        continue
    if y_tr_d.sum() < 5 or (len(y_tr_d) - y_tr_d.sum()) < 5:
        continue

    best_val_f1, best_clf, best_t = -1, None, 0.54

    for hp in HP_GRID:
        clf = LGBMClassifier(
            class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1,
            **hp
        )
        clf.fit(X_tr_d, y_tr_d)
        proba_va = clf.predict_proba(X_va_d)[:, 1]
        t, f1_val = best_threshold(y_va_d, proba_va)
        if f1_val > best_val_f1:
            best_val_f1, best_clf, best_t = f1_val, clf, t
            best_hp = hp

    tuned_models[domain]     = best_clf
    tuned_thresholds[domain] = best_t

    print(f"  {domain:<30}  best_val_F1={best_val_f1*100:.2f}%  t={best_t:.2f}  "
          f"leaves={best_hp['num_leaves']}  n_est={best_hp['n_estimators']}")

print(f"\nTuned domain models: {len(tuned_models)}")

In [ ]:
# Evaluate tuned models on test set
y_pred_tuned  = y_pred_base.copy().astype(float)
y_proba_tuned = proba_test_base.copy()

for domain, clf in tuned_models.items():
    te_mask = d_test == domain
    if te_mask.sum() < 30:
        continue
    proba_d = clf.predict_proba(X_test[te_mask])[:, 1]
    thresh  = tuned_thresholds[domain]
    y_pred_tuned[te_mask]  = (proba_d >= thresh).astype(int)
    y_proba_tuned[te_mask] = proba_d

res_tuned = metrics(y_test, y_pred_tuned.astype(int), y_proba_tuned)
print("=== TUNED DOMAIN MODELS — TEST ===")
for k, v in res_tuned.items():
    diff = v - res_base[k]
    print(f"  {k:<12}: {v*100:.2f}%  ({diff*100:+.2f} pp vs baseline)")

## 8. Refinement 4 — Domain as Feature (Universal + Domain OHE)

Instead of splitting the data, add one-hot-encoded domain labels to the universal feature matrix.
The model can then learn domain-specific weights without the sample-size penalty of full segmentation.

In [ ]:
print("=" * 80)
print("REFINEMENT 4: DOMAIN AS FEATURE")
print("=" * 80)

domain_ohe = pd.get_dummies(df['domain'], prefix='dom').reindex(X_all.index, fill_value=0)

X_train_aug = pd.concat([X_train, domain_ohe.loc[X_train.index]], axis=1)
X_val_aug   = pd.concat([X_val,   domain_ohe.loc[X_val.index]],   axis=1)
X_test_aug  = pd.concat([X_test,  domain_ohe.loc[X_test.index]],  axis=1)

# Align columns (test/val may have seen different OHE cols)
X_val_aug   = X_val_aug.reindex(columns=X_train_aug.columns, fill_value=0)
X_test_aug  = X_test_aug.reindex(columns=X_train_aug.columns, fill_value=0)

print(f"Augmented feature matrix: {X_train_aug.shape}")
print(f"Added {X_train_aug.shape[1] - X_train.shape[1]} domain OHE columns")

aug_clf = copy.deepcopy(BASE_CLF)
aug_clf.fit(X_train_aug, y_train)

proba_val_aug  = aug_clf.predict_proba(X_val_aug)[:, 1]
proba_test_aug = aug_clf.predict_proba(X_test_aug)[:, 1]

t_aug, f1_val_aug = best_threshold(y_val, proba_val_aug)
print(f"\nDomain-as-feature — val threshold: {t_aug:.2f}  val F1: {f1_val_aug*100:.2f}%")

y_pred_aug = (proba_test_aug >= t_aug).astype(int)
res_aug = metrics(y_test, y_pred_aug, proba_test_aug)

print("\n=== DOMAIN-AS-FEATURE — TEST ===")
for k, v in res_aug.items():
    diff = v - res_base[k]
    print(f"  {k:<12}: {v*100:.2f}%  ({diff*100:+.2f} pp vs baseline)")

## 9. Refinement 5 — Combined: Tuned + Calibrated Domain Models

Stack the best of each refinement: use per-domain **tuned** LightGBM, then **Platt-calibrate** on val,
then pick the threshold that maximises val F1. Fall back to the augmented universal model for small/missing domains.

In [ ]:
print("=" * 80)
print("REFINEMENT 5: TUNED + CALIBRATED DOMAIN MODELS")
print("=" * 80)

combined_models     = {}
combined_thresholds = {}

for domain in DOMAINS:
    tr_mask = d_train == domain
    va_mask = d_val   == domain

    X_tr_d, y_tr_d = X_train[tr_mask], y_train[tr_mask]
    X_va_d, y_va_d = X_val[va_mask],   y_val[va_mask]

    if len(X_tr_d) < 50 or len(X_va_d) < 30:
        continue
    if y_tr_d.sum() < 5 or (len(y_tr_d) - y_tr_d.sum()) < 5:
        continue

    # Use the already-tuned model from Refinement 3
    if domain not in tuned_models:
        continue

    clf_tuned = tuned_models[domain]

    # Platt-calibrate on val
    cal = CalibratedClassifierCV(clf_tuned, cv='prefit', method='sigmoid')
    cal.fit(X_va_d, y_va_d)

    proba_va_cal = cal.predict_proba(X_va_d)[:, 1]
    t_d, f1_val_d = best_threshold(y_va_d, proba_va_cal)

    combined_models[domain]     = cal
    combined_thresholds[domain] = t_d

    print(f"  {domain:<30}  val_F1={f1_val_d*100:.2f}%  t={t_d:.2f}")

print(f"\nCombined models: {len(combined_models)}")

In [ ]:
# Fall back to augmented universal for domains without a combined model
y_pred_combined  = y_pred_aug.copy().astype(float)   # start from domain-as-feature baseline
y_proba_combined = proba_test_aug.copy()

for domain, cal in combined_models.items():
    te_mask = d_test == domain
    if te_mask.sum() < 30:
        continue
    proba_d = cal.predict_proba(X_test[te_mask])[:, 1]
    thresh  = combined_thresholds[domain]
    y_pred_combined[te_mask]  = (proba_d >= thresh).astype(int)
    y_proba_combined[te_mask] = proba_d

res_combined = metrics(y_test, y_pred_combined.astype(int), y_proba_combined)
print("=== TUNED + CALIBRATED DOMAIN MODELS — TEST ===")
for k, v in res_combined.items():
    diff = v - res_base[k]
    print(f"  {k:<12}: {v*100:.2f}%  ({diff*100:+.2f} pp vs baseline)")

## 10. Per-Domain Breakdown

Which domains drive the gains? Where are we still losing to the universal model?

In [ ]:
rows = []
for domain in sorted(combined_models.keys()):
    te_mask = d_test == domain
    if te_mask.sum() < 30:
        continue

    y_te_d    = y_test[te_mask]
    f1_base_d = f1_score(y_te_d, y_pred_base[te_mask],          zero_division=0)
    f1_comb_d = f1_score(y_te_d, y_pred_combined[te_mask].astype(int), zero_division=0)

    rows.append(dict(
        Domain   = domain,
        N_test   = te_mask.sum(),
        Base_F1  = f1_base_d,
        Comb_F1  = f1_comb_d,
        Delta_pp = (f1_comb_d - f1_base_d) * 100,
    ))

breakdown_df = pd.DataFrame(rows).sort_values('Delta_pp', ascending=False)
print(breakdown_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(breakdown_df))
w = 0.35
ax.bar(x - w/2, breakdown_df['Base_F1']*100, w, label='Universal Baseline', alpha=0.8)
ax.bar(x + w/2, breakdown_df['Comb_F1']*100, w, label='Combined Domain Model', alpha=0.8)
ax.axhline(BASELINE_F1*100, color='red', ls='--', alpha=0.5, label=f'Overall baseline ({BASELINE_F1*100:.2f}%)')
ax.set_xticks(x)
ax.set_xticklabels(breakdown_df['Domain'], rotation=30, ha='right')
ax.set_ylabel('F1 (%)')
ax.set_title('Per-Domain F1: Universal Baseline vs Combined Domain Model')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Final Summary

In [ ]:
print("\n" + "=" * 80)
print("FINAL SUMMARY — ALL REFINEMENTS  (test set: 2018-2020, threshold tuned on val 2017)")
print("=" * 80)

all_results = [
    ("Universal LightGBM (val-tuned threshold)",     res_base),
    ("R1: Calibrated domain models",                  res_calib),
    ("R2: Feature-selected domain models (k=50)",     res_featsel),
    ("R3: Per-domain hyperparameter tuning",          res_tuned),
    ("R4: Domain-as-feature (OHE in universal)",      res_aug),
    ("R5: Tuned + Calibrated (combined best)",        res_combined),
]

print(f"\n{'Method':<50} {'F1':>8} {'vs Base':>10} {'ROC-AUC':>10}")
print("-" * 80)
for name, res in all_results:
    f1   = res.get('F1', 0)
    auc  = res.get('ROC-AUC', float('nan'))
    diff = f1 - BASELINE_F1
    marker = " ← BEST" if f1 == max(r.get('F1', 0) for _, r in all_results) else ""
    print(f"  {name:<48} {f1*100:>7.2f}%  {diff*100:>+7.2f} pp  {auc*100:>8.2f}%{marker}")

best_name, best_res = max(all_results, key=lambda x: x[1].get('F1', 0))
best_f1_test = best_res['F1']

print("\n" + "=" * 80)
print(f"BEST METHOD : {best_name}")
print(f"TEST F1     : {BASELINE_F1*100:.2f}% → {best_f1_test*100:.2f}%  "
      f"(+{(best_f1_test - BASELINE_F1)*100:.2f} pp)")
print("=" * 80)